<a href="https://colab.research.google.com/github/courbel/git-solo-tutorial/blob/main/Modeling_PennMutual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

filename = 'PennGVULReconstruction.xlsx'
search_path = '/content/drive/My Drive/Data'
file_path = None

for root, dirs, files in os.walk(search_path):
    for file in files:
        if file == filename:
            file_path = os.path.join(root, file)
            break
    if file_path:
        break

if file_path is None:
    raise FileNotFoundError(f"'{filename}' not found in the specified directory.")
else:
    df = pd.read_excel(file_path)

In [6]:
data = pd.read_excel(file_path, sheet_name="52 10%", usecols=[41, 42, 43, 44]).iloc[4:70]
data.set_index(data.columns[0], inplace=True)
data.index.name = "Year"
data.columns = [35, 45, 55]

approx_coi = pd.read_excel(file_path, sheet_name="COI Charges", usecols=[5,9]).iloc[0:91]
approx_coi.set_index(approx_coi.columns[0], inplace=True)
approx_coi.index.name = "Year"
approx_coi.columns = ["COI"]

In [10]:
def poly_regr_mult(data, degree, age, plan_duration):
  X = data.columns.values.reshape(-1, 1)
  predicted = {}

  for i in range(1, plan_duration + 1):
    y = data.loc[i].values

    pr = PolynomialFeatures(degree=degree)
    X_poly = pr.fit_transform(X)
    lr_2 = LinearRegression()
    lr_2.fit(X_poly, y)

    predicted[i] = lr_2.predict(pr.fit_transform([[age]]))[0]


  predicted_df = pd.DataFrame.from_dict(predicted, orient='index', columns=['Value'])
  predicted_df.index.name = 'Year'

  return predicted_df

In [11]:
def calc_error_mult(data, estimated, years):
  cum_sum = 0
  for i in range(1, years + 1):
    cum_sum += abs((data["COI"][i] - estimated["COI"][i]) / data["COI"][i])

  return cum_sum / years

In [ ]:
#for a 52 year old for example

min = 100
for i in range(100):
  estimated_data = poly_regr_mult(data, i, 52, 67)
  val = calc_error_avg(data, estimated_data, 67)
  if val < min:
    min = val
    degree = i

print(degree)

In [13]:
db = pd.read_excel(file_path, sheet_name="Death Benefit", usecols=[0, 1]).iloc[0:3]
db.set_index(db.columns[0], inplace=True)
db.index.name = "Year"
db.columns = ["Death Benefit"]

db_real = "some value"

In [ ]:
def poly_regr_single(data, degree, age):
  X = data.index.values.reshape(-1, 1)
  y = data.values

  pr = PolynomialFeatures(degree=degree)
  X_poly = pr.fit_transform(X)
  lr_2 = LinearRegression()
  lr_2.fit(X_poly, y)

  predicted = lr_2.predict(pr.fit_transform([[age]]))[0]

  return predicted[0]

In [ ]:
def calc_error_single(real, estimated):
  return abs((real - estimated) / real)

In [ ]:
min = 100
for i in range(100):
  estimated_data = poly_regr_single(data, i, 52)
  val = calc_error_single(db_real, estimated_data)
  if val < min:
    min = val
    degree = i

print(degree)

In [14]:
cp = pd.read_excel(file_path, sheet_name="Cost Per 1000", usecols=[0, 1]).iloc[0:3]
cp.set_index(cp.columns[0], inplace=True)
cp.index.name = "Year"
cp.columns = ["Cost Per 1000"]

cp100 = "some value"